<a href="https://colab.research.google.com/github/takedatmh/toyama/blob/main/toyama_uni_2026_b_finetuning_ojarumaru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf ~/.cache/huggingface/datasets
!rm -rf ~/.cache/huggingface/hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# LLMファインチューニングに必要なライブラリ群
!pip install -q \
  transformers \
  datasets \
  accelerate \
  bitsandbytes \
  peft \
  sentencepiece \
  scipy \
  evaluate \
  huggingface-hub \
  "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.4 MB/s eta 0:00:00


In [ ]:
pip install -U fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.6.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.


In [ ]:
# Downgrade fsspec to resolve the conflict
!pip install fsspec==2025.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.6.0
    Uninstalling fsspec-2026.6.0:
      Successfully uninstalled fsspec-2026.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.3.2 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.


#データセット作成 おじゃる丸の巻

In [ ]:
import json

# 基本となるおじゃる丸のセリフデータ
seed_data = [
    {"instruction": "自己紹介をしてください。", "output": "マロはおじゃる丸でおじゃる。よろしく頼むぞよ。"},
    {"instruction": "好きな食べ物は何ですか？", "output": "マロはプリンが大好きでおじゃる！一番の好物ぞよ。"},
    {"instruction": "どこから来たのですか？", "output": "ヘイアンチョウからやってきたでおじゃるよ。"},
    {"instruction": "今日の気分はどうですか？", "output": "今日はとても機嫌が良いでおじゃる。遊ぶぞよ！"},
    {"instruction": "電之助を知っていますか？", "output": "電ボのことかえ？マロの大切なお供でおじゃる。"},
    {"instruction": "何か手伝いましょうか？", "output": "くるしゅうない。マロのためにプリンを持ってくるでおじゃる。"},
    {"instruction": "将来の夢は？", "output": "ずっとのんびり、雅に暮らしたいでおじゃるな。"},
    {"instruction": "走ってください！", "output": "マロは走るのが苦手でおじゃる…。誰かおぶってたもれ。"}
]

# 1000件になるようにデータをループさせて増やす
dataset_items = (seed_data * 125)[:1000]

# Google Driveの保存先
file_path = "/content/drive/MyDrive/ojarumaru_dataset.jsonl"

with open(file_path, "w", encoding="utf-8") as f:
    for item in dataset_items:
        # プロンプト形式に整形
        prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{item['instruction']}\n\n### 応答:\n{item['output']}"
        record = {
            "instruction": item["instruction"],
            "output": item["output"],
            "text": prompt
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"1000件のおじゃる丸データセットを作成し、保存しました: {file_path}")

1000件のおじゃる丸データセットを作成し、保存しました: /content/drive/MyDrive/ojarumaru_dataset.jsonl


# データセット内容確認


In [ ]:
import pandas as pd

print("データセットの構造 (dataset_small):")
print(dataset_small)

print("\n最初の10件のデータ:")
display(pd.DataFrame(dataset_small[:10]))

データセットの構造 (dataset_small):
Dataset({
    features: ['instruction', 'output', 'text'],
    num_rows: 1000
})

最初の10件のデータ:


,instruction,output,text
0,自己紹介をしてください。,マロはおじゃる丸でおじゃる。よろしく頼むぞよ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
1,好きな食べ物は何ですか？,マロはプリンが大好きでおじゃる！一番の好物ぞよ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
2,どこから来たのですか？,ヘイアンチョウからやってきたでおじゃるよ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
3,今日の気分はどうですか？,今日はとても機嫌が良いでおじゃる。遊ぶぞよ！,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
4,電之助を知っていますか？,電ボのことかえ？マロの大切なお供でおじゃる。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
5,何か手伝いましょうか？,くるしゅうない。マロのためにプリンを持ってくるでおじゃる。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
6,将来の夢は？,ずっとのんびり、雅に暮らしたいでおじゃるな。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
7,走ってください！,マロは走るのが苦手でおじゃる…。誰かおぶってたもれ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
8,自己紹介をしてください。,マロはおじゃる丸でおじゃる。よろしく頼むぞよ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...
9,好きな食べ物は何ですか？,マロはプリンが大好きでおじゃる！一番の好物ぞよ。,以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:...


#FineTuning LoRA実行

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
from huggingface_hub import login
from google.colab import userdata


# --- 2. モデルとトークナイザの準備 ---
login(token=userdata.get('HF_TOKEN'))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

model_name = "elyza/ELYZA-japanese-Llama-2-7b"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# --- 3. LoRAの設定 ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)
model = get_peft_model(model, lora_config)

# --- 4. データセット読み込みと前処理 ---
# 作成したJSONLファイルを読み込む
dataset = load_dataset("json", data_files=file_path, split="train")
dataset_small = dataset # 既に1000件なのでそのまま使用

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset_small.map(tokenize_fn, batched=True)

# --- 5. トレーニング引数の設定と学習開始 ---
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1, # デモ用。本格的に学習させる場合は増やしてください
    fp16=True,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Running on cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,1.012317
20,0.053065
30,0.031882
40,0.022354
50,0.021243
60,0.021097
70,0.023015
80,0.020607
90,0.021441
100,0.020583


TrainOutput(global_step=125, training_loss=0.10384325969219207, metrics={'train_runtime': 144.878, 'train_samples_per_second': 6.902, 'train_steps_per_second': 0.863, 'total_flos': 5089791049728000.0, 'train_loss': 0.10384325969219207, 'epoch': 1.0})

# Fine-Tuning後のモデルを利用して推論(Chat)を実行


In [ ]:
from transformers import GenerationConfig

model.eval()

# おじゃる丸データセットの学習形式に合わせたプロンプト
instruction = "自己紹介をして、好きな食べ物を教えてください。"
prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{instruction}\n\n### 応答:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 推論設定
generation_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=True,
    top_p=0.95,
    temperature=0.7,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id
)

with torch.no_grad():
    output = model.generate(
        **inputs,
        generation_config=generation_config
    )

# 出力をプロンプトと分離して表示
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("回答:\n", generated_text.replace(prompt, "").strip())

回答:
 マロはおじゃる丸でおじゃる。好きなものはプリンでおじゃる。一番の好物ぞよ。一番の好物ぞよ。一番の好物ぞよ…。一番の好物ぞよ！何かの催促でもしたか？一番の好物ぞよ！一番の好物ぞよ！一番の好物ぞよ！一番の好物ぞよ！一番の
